In [ ]:
import ast
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from chef_classifier.data import (
    clean_training_data,
    create_train_val_split,
    load_training_data,
)
from chef_classifier.evaluation import calculate_accuracy
from chef_classifier.features import combine_text_fields
from chef_classifier.models import build_tfidf_svc_pipeline

In [ ]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test-no-labels.csv"

In [ ]:
train = load_training_data(TRAIN_PATH)
test = pd.read_csv(TEST_PATH, sep=";")

In [ ]:
train_clean = clean_training_data(train)
train_clean.shape

In [ ]:
train_df, val_df = create_train_val_split(train_clean)

In [ ]:
X_train = combine_text_fields(train_df, ["description"])
X_val = combine_text_fields(val_df, ["description"])

model = build_tfidf_svc_pipeline()
model.fit(X_train, train_df["chef_id"])

predictions = model.predict(X_val)

accuracy = calculate_accuracy(val_df["chef_id"], predictions)
accuracy

In [ ]:
svc = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("classifier", SVC(kernel="linear")),
    ]
)
svc.fit(X_train, train_df["chef_id"])

svc_predictions = svc.predict(X_val)
accuracy_score(val_df["chef_id"], svc_predictions)

In [ ]:
print(classification_report(val_df["chef_id"], predictions))

In [ ]:
train_df["description"].duplicated().sum(), val_df["description"].duplicated().sum()

In [ ]:
val_df["description"].isin(train_df["description"]).sum()

In [ ]:
duplicate_description_mask = val_df["description"].isin(train_df["description"])

duplicate_description_mask.sum()

In [ ]:
clean_val_accuracy = accuracy_score(
    val_df.loc[~duplicate_description_mask, "chef_id"],
    predictions[~duplicate_description_mask],
)

clean_val_accuracy

In [ ]:
overlap_accuracy = accuracy_score(
    val_df.loc[duplicate_description_mask, "chef_id"],
    predictions[duplicate_description_mask],
)

overlap_accuracy

- TF-IDF features extracted only from the description field perform surprisingly strongly:
  - linear-kernel SVC: 71.9%
  - LinearSVC: 73.4%
- Seven validation examples have descriptions that also occur in the training subset.
- The model predicts all seven of these overlapping descriptions correctly.
- Excluding those overlapping examples reduces validation accuracy only slightly, from 73.4% to 73.1%.
- Therefore, repeated descriptions introduce a small amount of leakage but do not explain the high overall performance.

In [ ]:
val_results = val_df.copy()

val_results["predicted_chef"] = predictions
val_results["correct"] = (
    val_results["chef_id"] == val_results["predicted_chef"]
)

val_results.head()

In [ ]:
val_results["correct"].value_counts()

In [ ]:
val_results["correct"].mean()

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    val_results["chef_id"],
    val_results["predicted_chef"],
)

In [ ]:
confusion_table = pd.crosstab(
    val_results["chef_id"],
    val_results["predicted_chef"],
    rownames=["True chef"],
    colnames=["Predicted chef"],
)

confusion_table

In [ ]:
errors = val_results[~val_results["correct"]].copy()

errors.shape

In [ ]:
errors[
    [
        "chef_id",
        "predicted_chef",
        "recipe_name",
        "description",
        "ingredients",
        "tags",
    ]
].sample(10, random_state=42)

In [ ]:
confusion_pairs = (
    errors.groupby(["chef_id", "predicted_chef"])
    .size()
    .sort_values(ascending=False)
)

confusion_pairs.head(10)

In [ ]:
pair_errors = errors[
    (errors["chef_id"] == 3288)
    & (errors["predicted_chef"] == 5060)
]

pair_errors[
    [
        "recipe_name",
        "description",
        "ingredients",
        "tags",
    ]
].head(10)

In [ ]:
correct_3288 = val_results[
    (val_results["chef_id"] == 3288)
    & (val_results["correct"])
]

correct_5060 = val_results[
    (val_results["chef_id"] == 5060)
    & (val_results["correct"])
]

In [ ]:
val_results["description_length"] = (
    val_results["description"].str.len()
)

val_results.groupby("correct")[
    "description_length"
].describe()

In [ ]:
val_results["description_words"] = (
    val_results["description"].str.split().str.len()
)

val_results.groupby("correct")[
    "description_words"
].describe()

In [ ]:
accuracy_by_chef = (
    val_results.groupby("chef_id")["correct"]
    .mean()
    .sort_values()
)

accuracy_by_chef

Classification difficulty varies substantially between chefs:

- chef 1533: 57.5%
- chef 3288: 57.8%
- chef 6357: 74.0%
- chef 5060: 77.6%
- chef 8688: 79.3%
- chef 4470: 83.8%

The errors are not uniformly distributed. Some of the most frequent confusion directions are, this suggests that some chefs use more similar vocabulary or description styles than others.

Incorrectly classified recipes tend to have shorter descriptions:

- incorrect predictions: about 20.4 words on average
- correct predictions: about 32.0 words on average

The corresponding median lengths are approximately 18** and 23.5 words.

This suggests that short or generic descriptions provide less lexical and stylistic evidence for identifying the chef.

Several misclassified examples have very short or generic descriptions, such as phrases equivalent to "this is so good", "for a hot day", or short comments about serving suggestions.

In such cases, useful identifying information may exist in other fields such as:

- recipe_name
- ingredients
- tags
- steps

The next experiments should therefore test whether adding these fields improves performance, particularly for chefs 1533 and 3288.

To measure the contribution of each field, the following experiments keep the validation split and LinearSVC classifier fixed while changing only the text fields used to build the TF-IDF representation.

In [ ]:
def evaluate_text_fields(fields: list[str]) -> float:
    train_text = combine_text_fields(train_df, fields)
    val_text = combine_text_fields(val_df, fields)

    model = build_tfidf_svc_pipeline()
    model.fit(train_text, train_df["chef_id"])

    predictions = model.predict(val_text)

    return calculate_accuracy(val_df["chef_id"], predictions)

In [ ]:
experiments = {
    "description": ["description"],
    "description + recipe_name": ["description", "recipe_name"],
    "description + ingredients": ["description", "ingredients"],
    "description + tags": ["description", "tags"],
    "description + steps": ["description", "steps"],
    "all text fields": [
        "description",
        "recipe_name",
        "ingredients",
        "tags",
        "steps",
    ],
}

results = {}

for name, fields in experiments.items():
    results[name] = evaluate_text_fields(fields)

results

In [ ]:
results_df = (
    pd.DataFrame.from_dict(
        results,
        orient="index",
        columns=["accuracy"],
    )
    .sort_values("accuracy", ascending=False)
)

results_df

In [ ]:
tag_only_accuracy = evaluate_text_fields(["tags"])
tag_only_accuracy

In [ ]:
train_df.groupby("chef_id")["tags"].apply(
    lambda x: x.head(3).tolist()
)

In [ ]:
train_tags = train_df[["chef_id", "tags"]].copy()

train_tags["tags"] = train_tags["tags"].apply(ast.literal_eval)

train_tags.head()

In [ ]:
exploded_tags = (
    train_tags
    .explode("tags")
    .reset_index(drop=True)
)

exploded_tags.head()

In [ ]:
tag_counts = (
    exploded_tags.groupby(["chef_id", "tags"])
    .size()
    .reset_index(name="count")
)

tag_counts.head()

In [ ]:
top_tags_by_chef = (
    tag_counts.sort_values(
        ["chef_id", "count"],
        ascending=[True, False],
    )
    .groupby("chef_id")
    .head(15)
)

top_tags_by_chef

In [ ]:
tag_chef_table = pd.crosstab(
    exploded_tags["tags"],
    exploded_tags["chef_id"],
)

tag_chef_table

In [ ]:
tag_dominance = pd.DataFrame({
    "total_count": tag_chef_table.sum(axis=1),
    "dominant_chef": tag_chef_table.idxmax(axis=1),
    "dominant_count": tag_chef_table.max(axis=1),
})

tag_dominance["dominance_ratio"] = (
    tag_dominance["dominant_count"]
    / tag_dominance["total_count"]
)

In [ ]:
tag_dominance[
    tag_dominance["total_count"] >= 10
].sort_values(
    ["dominance_ratio", "total_count"],
    ascending=[False, False],
).head(30)

### Tag analysis

The high predictive performance of tags can be explained by strong associations between certain tags and individual chefs.

Several tags are almost exclusive to a single chef. Examples include:

- scandinavian: 17/17 occurrences belong to chef 4470
- danish: 14/14 belong to chef 4470
- british-columbian: 242/243 belong to chef 5060
- pacific-northwest: 99/100 belong to chef 5060
- ontario: 67/69 belong to chef 1533
- indian: 212/223 belong to chef 6357

Other tags such as ramadan, curries, gluten-free, asian, and oamc-freezer-make-ahead also show strong chef-specific concentration.

This indicates that the classifier is not relying only on linguistic writing style. It can also identify chefs from the types of recipes they tend to create and the categories associated with those recipes.

This appears to be a legitimate predictive signal in the supplied dataset, although it may also reflect dataset-specific author preferences. Therefore, very high local validation performance may not generalize equally well if the distribution of recipe categories changes in the test set.

## Cross-validation

The previous experiments used a single stratified 80/20 train-validation split. To obtain a more reliable estimate of model performance, the best-performing text configuration is evaluated using stratified k-fold cross-validation.

In [ ]:
fields = ["description", "tags"]

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = combine_text_fields(train_clean, fields)
y = train_clean["chef_id"]

model = build_tfidf_svc_pipeline()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy",
)

scores

In [ ]:
scores.mean(), scores.std()

## TF-IDF Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

pipeline = build_tfidf_svc_pipeline()

param_grid = {
    "tfidf__ngram_range": [
        (1, 1),
        (1, 2),
    ],
    "tfidf__min_df": [
        1,
        2,
        3,
    ],
    "tfidf__sublinear_tf": [
        False,
        True,
    ],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

grid_search.fit(X, y)

In [ ]:
grid_search.best_score_

In [ ]:
grid_search.best_params_

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X_numeric = train_clean[["n_ingredients"]]
y = train_clean["chef_id"]

numeric_model = LogisticRegression(
    max_iter=1000,
)

numeric_scores = cross_val_score(
    numeric_model,
    X_numeric,
    y,
    cv=cv,
    scoring="accuracy",
)

numeric_scores.mean(), numeric_scores.std()

In [ ]:
structured_df = train_clean.copy()

structured_df["year"] = pd.to_datetime(
    structured_df["data"],
    format="%d/%m/%Y",
).dt.year

X_year = structured_df[["year"]]

year_scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_year,
    y,
    cv=cv,
    scoring="accuracy",
)

year_scores.mean(), year_scores.std()

In [ ]:
X_structured = structured_df[
    ["year", "n_ingredients"]
]

structured_scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_structured,
    y,
    cv=cv,
    scoring="accuracy",
)

structured_scores.mean(), structured_scores.std()

In [ ]:
experiment_df = train_clean.copy()

experiment_df["combined_text"] = combine_text_fields(
    experiment_df,
    ["description", "tags"],
)

experiment_df["year"] = pd.to_datetime(
    experiment_df["data"],
    format="%d/%m/%Y",
).dt.year

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "text",
            TfidfVectorizer(),
            "combined_text",
        ),
        (
            "numeric",
            StandardScaler(),
            ["year", "n_ingredients"],
        ),
    ]
)

combined_model = Pipeline(
    [
        ("features", preprocessor),
        ("classifier", LinearSVC()),
    ]
)

combined_scores = cross_val_score(
    combined_model,
    experiment_df,
    y,
    cv=cv,
    scoring="accuracy",
)

combined_scores.mean(), combined_scores.std()

In [ ]:
experiment_results = [
    {
        "experiment": "description + tags (default TF-IDF)",
        "cv_mean_accuracy": scores.mean(),
        "cv_std_accuracy": scores.std(),
    },
    {
        "experiment": "description + tags (tuned TF-IDF)",
        "cv_mean_accuracy": grid_search.best_score_,
        "cv_std_accuracy": None,
    },
    {
        "experiment": "n_ingredients",
        "cv_mean_accuracy": numeric_scores.mean(),
        "cv_std_accuracy": numeric_scores.std(),
    },
    {
        "experiment": "year",
        "cv_mean_accuracy": year_scores.mean(),
        "cv_std_accuracy": year_scores.std(),
    },
    {
        "experiment": "year + n_ingredients",
        "cv_mean_accuracy": structured_scores.mean(),
        "cv_std_accuracy": structured_scores.std(),
    },
    {
        "experiment": "description + tags + structured",
        "cv_mean_accuracy": combined_scores.mean(),
        "cv_std_accuracy": combined_scores.std(),
    },
]

results_df = pd.DataFrame(experiment_results).sort_values(
    "cv_mean_accuracy",
    ascending=False,
)

results_df

year and n_ingredients are weak alone , no useful improvement when added to strong text features.

Therefore, the predictive signal in this dataset appears to come primarily from textual features, particularly description and `tags.
